# Expected Calibration Error

A function that measures how well the predicted probabilities of a binary classifier are calibrated.

In [1]:
from mlxtend.evaluate import expected_calibration_error

> from mlxtend.evaluate import expected_calibration_error

## Overview

Accuracy tells you how often a classifier is right, but not whether its predicted probabilities can be trusted. A model is *well calibrated* if, among all samples for which it predicts a positive-class probability of about $p$, a fraction of about $p$ are actually positive.

The **Expected Calibration Error (ECE)** groups predictions into $M$ equal-width probability bins $B_1, \dots, B_M$ and averages the gap between the observed fraction of positives and the mean predicted probability in each bin, weighted by bin size:

$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} \left| \text{frac\_pos}(B_m) - \text{conf}(B_m) \right|$$

An ECE of 0 means perfect calibration; values closer to 1 mean worse calibration.

### References

- Naeini, M. P., Cooper, G., & Hauskrecht, M. (2015). Obtaining well calibrated probabilities using Bayesian binning. *AAAI*.
- Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks. *ICML*.

## Example 1 - A small hand-checkable case

In [2]:
import numpy as np
from mlxtend.evaluate import expected_calibration_error

y = np.array([0, 1, 1, 1])
p = np.array([0.1, 0.9, 0.8, 0.3])

expected_calibration_error(y, p, n_bins=2)

0.22499999999999995

With two bins, the first holds the predictions 0.1 and 0.3 (observed positive rate 0.5, mean probability 0.2, gap 0.3) and the second holds 0.9 and 0.8 (observed 1.0, mean 0.85, gap 0.15). Each bin has half the samples, so ECE = 0.5 * 0.3 + 0.5 * 0.15 = 0.225.

## Example 2 - Comparing the calibration of two classifiers

Naive Bayes is known to produce over-confident probabilities, while logistic regression is usually reasonably calibrated.

In [3]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

X, y = make_classification(
    n_samples=5000, n_features=20, n_informative=5, n_redundant=10, random_state=123
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=123)

for name, clf in [
    ("Logistic regression", LogisticRegression(max_iter=1000)),
    ("Gaussian naive Bayes", GaussianNB()),
]:
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)[:, 1]
    print(f"{name}: ECE = {expected_calibration_error(y_test, prob):.3f}")

Logistic regression: ECE = 0.015
Gaussian naive Bayes: ECE = 0.092


## API